# Strategic Sourcing Intelligence

**Question:** Where do publicly disclosed procurement awards suggest supplier concentration worth investigating?

**Source:** [World Bank Group corporate procurement awards](https://financesone.worldbank.org/d/DS00028). Download the official CSV to `../data/raw/world_bank_awards.csv`.

**Caution:** Awards are not payments, unit prices, delivery performance, or P&G spend.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.pipeline import load_awards, category_metrics, sensitivity, run
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data validation
Stop if the authentic source CSV is missing. We never fill missing source variables with synthetic data.

In [ ]:
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
source = root / "data/raw/world_bank_awards.csv"
assert source.exists(), "Download the official CSV first; see README.md"
awards, quality = load_awards(source)
quality

## 2. Supplier concentration by category and calendar year
HHI is the sum of squared supplier award-value shares multiplied by 10,000. It measures concentration **within this published dataset**, not market share or risk probability.

In [ ]:
metrics = category_metrics(awards)
metrics.head(15)

## 3. Inspect a category and challenge the interpretation
Category and contract scope differences can dominate observed award-value variation. Do not call this a price comparison.

In [ ]:
latest = int(awards.year.max())
eligible = metrics[(metrics.year == latest) & metrics.eligible_for_comparison]
if not eligible.empty:
    chosen = eligible.sort_values("awarded_usd", ascending=False).iloc[0].category
    sub = awards[(awards.year == latest) & (awards.category == chosen)]
    print("Selected category:", chosen)
    display(sub.groupby("supplier").award_value.agg(["sum","count"]).sort_values("sum", ascending=False).head(15))
else:
    print("No category meets the minimum comparison screen")

## 4. Sensitivity, not forecast
The hypothetical shift assumes suppliers can be qualified and awarded work. Neither capability nor cost is observed.

In [ ]:
if not eligible.empty:
    shares = sub.groupby("supplier").award_value.sum()
    shares = shares / shares.sum()
    display(sensitivity(shares))

## 5. Export reproducible management materials
Review the evidence brief and charts in `outputs/` before making any recommendations.

In [ ]:
run(source, root / "outputs")

## Executive interpretation checklist

1. Is supplier concentration real, or driven by contract type and category scope?
2. Could supplier aliases or corporate parents change the rankings?
3. What is the effect of the disclosure threshold change?
4. Which additional operational data would a purchasing manager require?
5. What conclusions are descriptive rather than causal?